In [1]:
import os
import pandas as pd
import numpy as np

#import datetime
#import json

import matplotlib.pyplot as plt
%matplotlib inline
#%timeit

In [2]:
#--- Folders and Directories

data_folder = r"..\data\afterstudy"
test_data = r"..\data"
output_folder = r"..\data\outputs"

OBS_2015_csv = r"Final 2015 Onboard Transit Passenger Survey.xlsx"
OBS_2023 = r"od_20240422_sandagca_weighted_combined_draftfinal_noPII.xlsx"
sheet_data = r"OD_RESULTS"

MTS_APC_file = r"request_20240418_mts_normalized_apc_stoplist.xlsx"
sheet_APC = r"NORMALIZED_APC"
sheet_farebox = r"FAREBOX_ROUTES"

NCTD_APC_file = r"request_20240507_sandag_NCTD_projectmaterials.xlsx"
sheet_NCTD_APC = r"APC"

In [3]:
srvy_filename = 'survey.csv'
unique_wts_filename = 'unique_weight_id.csv'
targets_filesname = 'obs_targets.csv'
importance_vect_filename = 'importance_vect.csv'
incidence_matrix_filename = 'incidence_mat.csv'

In [4]:
# These are the MTS routes where we do not have APC data,
# but farebox data is available. 
# These are either peak-direction only rural routes or circular routes.

farebox_routes = ['888', 
                  '891', 
                  '892', 
                  '906/907', 
                  '972', 
                  '973', 
                  '974', 
                  '978', 
                  '979']

# rail routes trolley, coaster, and sprinter, which will use the stop level targets
rail_routes = ['398', '399', 'Blue', 'Green', 'Orange']

In [6]:
obs_keep_cols = ['ROUTE_DIRECTION',
                 'TIME_PERIOD']

apc_keep_cols = ['ETC_ROUTE_ID',
                 'ON_EXPAND_AM',
                 'ON_EXPAND_Midday',
                 'ON_EXPAND_PM Peak',
                 'ON_EXPAND_PM Late',
                 'OFF_EXPAND_AM',
                 'OFF_EXPAND_Midday',
                 'OFF_EXPAND_PM Peak',
                 'OFF_EXPAND_PM Late']

NCTD_apc_keep_cols = ['route_short_name',
                      'ETC_STOP_ID',
                      'ON_EXPAND_AM',
                      'ON_EXPAND_Midday',
                      'ON_EXPAND_PM Peak',
                      'ON_EXPAND_PM Late',
                      'OFF_NORMALIZED_AM',
                      'OFF_NORMALIZED_Midday',
                      'OFF_NORMALIZED_PM Peak',
                      'OFF_NORMALIZED_PM Late']


srvy_use_cols = ['ID', 
                 'ROUTE_DIRECTION[Code]', 
                 'ROUTE_DIRECTION', 
                 'DIRECTION', 
                 'TIME_PERIOD', 
                 'UNLINKED_WGHT_FCTR']


rename_cols = {
    'TIME_PERIOD': 'TOD',
    'ROUTE_NUMBER': 'Route',
    'ROUTE_NAME': 'RtName',
    'DIRECTION_NAME': 'Dir',
    'SERVICE_CODE': 'Agency',
    'SUM_PASSENGERS_ON': 'ONs',
    'SUM_PASSENGERS_OFF': 'OFFs'    
}

time_periods_remap = {
    'AM': 'AM',
    'Midday': 'MIDDAY',
    'PM Peak': 'PM',
    'PM Late': 'EVE',
    'Other': 'OTH'
}


line_brds_grpby = ['ROUTE_NUMBER', 'DIRECTION_NAME', 'TIME_PERIOD']
agg_scheme = {
    'ROUTE_NAME': 'first',
    'SERVICE_CODE': 'first',
    'SUM_PASSENGERS_ON': 'sum',
    'SUM_PASSENGERS_OFF': 'sum',
}

def get_count():
    """
    """
    counts_2019 = pd.read_excel(os.path.join(raw_data, Midcoast_data), 
                                sheet_name = sheet_2019,
                                usecols = keep_cols,
                                engine='openpyxl')
    counts_2020 = pd.read_excel(os.path.join(raw_data, Midcoast_data),
                                sheet_name = sheet_2020,
                                usecols = keep_cols,
                                engine='openpyxl')
    for dframe in (counts_2019, counts_2020):
        dframe['TIME_PERIOD'] = dframe['TIME_PERIOD'].map(time_periods_remap)
        dframe['DIRECTION_NAME'] = dframe['DIRECTION_NAME'].map(dir_remap)


    def agg_line_brds(dframe, cols = line_brds_grpby, agg_scheme = agg_scheme):
        dframe_grouped = dframe.groupby(cols).agg(agg_scheme)
        dframe_grouped = dframe_grouped.reset_index()
        return dframe_grouped

    line_brds_2019, line_brds_2020 = (agg_line_brds(dframe) for dframe in (counts_2019, counts_2020))

    for dframe in (line_brds_2019, line_brds_2020):
        dframe['key'] = dframe['ROUTE_NUMBER'].astype('str') + dframe['DIRECTION_NAME'] + dframe['TIME_PERIOD']
        
    counts_recent = pd.merge(left = line_brds_2019, right = line_brds_2020,
                             how='outer',
                             on='key',
                             suffixes=('', '20')
                            )
    
    rename_cols20 = {
        'TIME_PERIOD20': 'TOD20',
        'ROUTE_NUMBER20': 'Route20',
        'ROUTE_NAME20': 'RtName20',
        'DIRECTION_NAME20': 'Dir20',
        'SERVICE_CODE20': 'Agency20',
        'SUM_PASSENGERS_ON20': 'ONs20',
        'SUM_PASSENGERS_OFF20': 'OFFs20'    
    }
    counts_recent = counts_recent.rename(columns=rename_cols)
    counts_recent = counts_recent.rename(columns=rename_cols20)

    return counts_recent


In [7]:
survey_2023 = pd.read_excel(os.path.join(data_folder, OBS_2023),
                            sheet_name = sheet_data,
                            engine='openpyxl')

In [8]:
survey_2023['AGENCY'] = survey_2023['ROUTE_DIRECTION[Code]'].str.split('_').str.get(0)
survey_2023['ROUTE_NUMBER'] = survey_2023['ROUTE_DIRECTION[Code]'].str.split('_').str.get(2)
survey_2023['BOARDING_ID']  = survey_2023['STOP_ON [CLNTID]'].str.split('_').str.get(4)
survey_2023['ALIGHTING_ID'] = survey_2023['STOP_OFF [CLNTID]'].str.split('_').str.get(4)
survey_2023['DIRECTION']    = survey_2023['ROUTE_DIRECTION[Code]'].str.split('_').str.get(3)

# for routes that only farebox data is available, update the route name and direction
# to match counts
survey_2023['ROUTE_NUMBER'] = \
    np.where(survey_2023['ROUTE_NUMBER'].isin(['906', '907']), '906/907', survey_2023['ROUTE_NUMBER'])

survey_2023['DIRECTION'] = \
    np.where(survey_2023['ROUTE_NUMBER'].isin(farebox_routes), '00', survey_2023['DIRECTION'])

survey_2023.loc[survey_2023['ALIGHTING_ID'].isnull(), 'ALIGHTING_ID'] = '00000'

In [9]:
# to count the number of route x direction x time_period combinations from the survey
# 888, include farebox routes (1 direction), and all rail routes

survey_line_groups = survey_2023.groupby(['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD']).agg({'ID':'count'})
survey_line_groups = survey_line_groups.reset_index()
survey_line_groups.to_csv(os.path.join(output_folder, "checks", "line_level_boarding_groups.csv"), index=False)
survey_line_groups

,ROUTE_NUMBER,DIRECTION,TIME_PERIOD,ID
0,1,00,AM,84
1,1,00,EVE,28
2,1,00,MIDDAY,133
3,1,00,PM,59
4,1,01,AM,69
...,...,...,...,...
883,Orange,00,PM,277
884,Orange,01,AM,282
885,Orange,01,EVE,206
886,Orange,01,MIDDAY,540


In [10]:
rail_line_groups = survey_2023[survey_2023['ROUTE_NUMBER'].isin(rail_routes)]
rail_line_groups = rail_line_groups.groupby(['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD']).agg({'ID':'count'})
rail_line_groups = rail_line_groups.reset_index()
rail_line_groups.to_csv(os.path.join(output_folder, "checks", "rail_line_level_boarding_groups.csv"), index=False)
rail_line_groups

,ROUTE_NUMBER,DIRECTION,TIME_PERIOD,ID
0,398,00,AM,26
1,398,00,EVE,15
2,398,00,MIDDAY,69
3,398,00,PM,99
4,398,01,AM,35
5,398,01,EVE,9
6,398,01,MIDDAY,75
7,398,01,PM,64
8,399,00,AM,57
9,399,00,EVE,61


In [12]:
# to count the number of [route x direction x time_period x boarding x alighting] combinations
# for rail lines, from the survey
# 4916 
rail_records = survey_2023[survey_2023['ROUTE_NUMBER'].isin(rail_routes)]
stop_grps = rail_records.groupby(['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD', 
                                 'BOARDING_ID', 'ALIGHTING_ID']).agg({'ID': 'count'}).reset_index()
stop_grps.to_csv(os.path.join(output_folder, "checks", "stop_boarding_groups.csv"), index=False)
stop_grps[stop_grps['ROUTE_NUMBER']=='Blue']

,ROUTE_NUMBER,DIRECTION,TIME_PERIOD,BOARDING_ID,ALIGHTING_ID,ID
446,Blue,00,AM,75000,00000,2
447,Blue,00,AM,75000,75002,4
448,Blue,00,AM,75000,75004,36
449,Blue,00,AM,75000,75006,9
450,Blue,00,AM,75000,75008,34
...,...,...,...,...,...,...
2765,Blue,01,PM,99885,75000,5
2766,Blue,01,PM,99885,75003,1
2767,Blue,01,PM,99885,75007,3
2768,Blue,01,PM,99885,75015,1


In [13]:
stop_grps['ROUTE_NUMBER'].unique()

array(['398', '399', 'Blue', 'Green', 'Orange'], dtype=object)

In [ ]:
MTS_routes = survey_2023[survey_2023['AGENCY'] == "MTS"]['ROUTE_NUMBER'].nunique()
MTS_routes

In [ ]:
NCT_routes = survey_2023[survey_2023['AGENCY'] == "NCT"]['ROUTE_NUMBER'].nunique()
NCT_routes

In [14]:
survey_2023['subkey1'] = ""
survey_2023['subkey2'] = ""
survey_2023['subkey3'] = ""
# Subkey1 is line_dir_tod
survey_2023['subkey1'] = (survey_2023['ROUTE_NUMBER'].astype('str') + '_' 
                     + survey_2023['DIRECTION'].astype('str') + '_'
                     + survey_2023['TIME_PERIOD']
                    )

# Subkey2 is line_dir_tod_brdStation only for trolleys: rt = (510, 520, 530)

survey_2023.loc[survey_2023['ROUTE_NUMBER'].isin(rail_routes), 'subkey2'] = (survey_2023['ROUTE_NUMBER'].astype('str') + '_' 
                     + survey_2023['DIRECTION'].astype('str') + '_'
                     + survey_2023['TIME_PERIOD'] + '_'
                     + survey_2023['BOARDING_ID'].astype('str') + '_'
                     + survey_2023['ALIGHTING_ID'].astype('str')
                    )

survey_2023.loc[survey_2023['ROUTE_NUMBER'].isin(rail_routes), 'subkey2_Ons'] = (survey_2023['ROUTE_NUMBER'].astype('str') + '_' 
                     + survey_2023['DIRECTION'].astype('str') + '_'
                     + survey_2023['TIME_PERIOD'] + '_'
                     + survey_2023['BOARDING_ID'].astype('str')
                    )

survey_2023.loc[survey_2023['ROUTE_NUMBER'].isin(rail_routes), 'subkey2_Offs'] = (survey_2023['ROUTE_NUMBER'].astype('str') + '_' 
                     + survey_2023['DIRECTION'].astype('str') + '_'
                     + survey_2023['TIME_PERIOD'] + '_'
                     + survey_2023['ALIGHTING_ID'].astype('str')
                    )

# Unique ID UID
select_rows = survey_2023['ROUTE_NUMBER'].isin(rail_routes)
survey_2023['UID'] = survey_2023['subkey1']
survey_2023['UID'] = np.where(select_rows, survey_2023['subkey2'], survey_2023['subkey1'])
                  
cols = srvy_use_cols + ['BOARDING_ID', 'ALIGHTING_ID', 'UID', 'subkey1', 'subkey2', 'subkey2_Ons','subkey2_Offs']
survey_2023.loc[survey_2023['ROUTE_NUMBER'].isin(rail_routes), cols]

survey_2023[cols].to_csv(os.path.join(output_folder, 'checks', srvy_filename))

In [15]:
key1 = survey_2023['subkey1'].nunique()
key2 = survey_2023['subkey2'].nunique()   # this is rail route x time_period x on/off combinations, includes empty value
uid =  survey_2023['UID'].nunique()
print("{}, {}, {}".format(key1, key2, uid))

888, 5053, 5900


In [16]:
subkey2_unique = survey_2023['subkey2'].unique()
subkey2_unique

array(['', '398_00_AM_28007_28000', '398_00_AM_28007_28005', ...,
       'Orange_01_EVE_75039_75068', 'Orange_01_EVE_75027_75068',
       'Orange_01_EVE_75088_75087'], dtype=object)

#### 2023 APC Counts

In [17]:
MTS_APC_raw = pd.read_excel(os.path.join(data_folder, MTS_APC_file), 
                            sheet_name = sheet_APC,
                            usecols = apc_keep_cols + ['stop_id'],
                            engine='openpyxl')

# remove the records that don't have APC data, these routes will use farebox data
MTS_APC_raw = MTS_APC_raw[MTS_APC_raw['ON_EXPAND_AM'] != '-']
MTS_APC_raw

,stop_id,ETC_ROUTE_ID,ON_EXPAND_AM,ON_EXPAND_Midday,ON_EXPAND_PM Peak,ON_EXPAND_PM Late,OFF_EXPAND_AM,OFF_EXPAND_Midday,OFF_EXPAND_PM Peak,OFF_EXPAND_PM Late
0,94048,MTS_1_1_00,23.558929,77.483855,69.94879,34.299582,0,0,0,0
1,13510,MTS_1_1_00,1.60755,5.191548,12.161147,1.394807,2.617978,1.559659,1.512765,1.781763
2,10478,MTS_1_1_00,2.60534,12.459715,13.048822,4.247823,1.00263,5.84872,4.716268,2.099935
3,13391,MTS_1_1_00,0.886924,5.710703,5.858655,2.282412,0.222807,2.079545,2.313641,0.57271
4,10106,MTS_1_1_00,2.106445,7.397956,5.237282,1.648409,0.278508,2.209516,2.847558,0.954516
...,...,...,...,...,...,...,...,...,...,...
5724,10094,MTS_1_992_01,0,0.66324,0,0.379501,3.139161,7.322896,4.135768,6.354563
5725,10099,MTS_1_992_01,0.860511,1.54756,0,2.150503,2.643504,9.921343,7.385301,3.574442
5726,13315,MTS_1_992_01,6.884086,19.8972,2.098724,2.656504,3.249307,19.370242,5.760534,3.839215
5727,12782,MTS_1_992_01,0.728124,0,0,0.506001,6.223249,30.70892,21.121959,26.344959


In [18]:
MTS_APC_data = pd.melt(MTS_APC_raw, id_vars = ['ETC_ROUTE_ID',
                                               'stop_id'])

MTS_APC_data['variable'] = MTS_APC_data['variable'].str.replace('EXPAND_','')
MTS_APC_data[['boarding','TIME_PERIOD']] = MTS_APC_data['variable'].str.split('_', expand=True)

MTS_APC_data[['AGENCY', 'AGENCY_CODE','ROUTE_NUMBER', 'DIRECTION']] = MTS_APC_data['ETC_ROUTE_ID'].str.split('_', expand=True)

In [19]:
MTS_APC_data = pd.pivot_table(MTS_APC_data, 
                              values="value", 
                              index=['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD', 'stop_id'],
                              columns = ['boarding'], 
                              aggfunc=np.sum, 
                              fill_value=0)
MTS_APC_data = MTS_APC_data.reset_index()

In [20]:
MTS_APC_data['TIME_PERIOD'] = MTS_APC_data['TIME_PERIOD'].map(time_periods_remap)
MTS_APC_data['TIME_PERIOD'].unique()

array(['AM', 'MIDDAY', 'EVE', 'PM'], dtype=object)

In [59]:
MTS_APC_data

boarding,ROUTE_NUMBER,DIRECTION,TIME_PERIOD,stop_id,OFF,ON
0,1,00,AM,10106,0.278508,2.106445
1,1,00,AM,10111,0.167105,1.330387
2,1,00,AM,10114,0.278508,0.443462
3,1,00,AM,10129,0.501315,2.051013
4,1,00,AM,10139,0.779823,7.594290
...,...,...,...,...,...,...
21239,Orange,01,PM,75090,206.995348,56.878722
21240,Orange,01,PM,75092,131.037055,88.318015
21241,Orange,01,PM,75102,730.903574,271.073908
21242,Orange,01,PM,75109,136.436934,0.000000


In [21]:
MTS_farebox_raw = pd.read_excel(os.path.join(data_folder, MTS_APC_file), 
                                sheet_name = sheet_farebox,
                                engine='openpyxl')

In [22]:
MTS_farebox_raw[['AGENCY', 'AGENCY_CODE', 'ROUTE_NUMBER']] = MTS_farebox_raw['ETC_ROUTE_ID'].str.split('_', expand=True)

MTS_farebox_count = pd.melt(MTS_farebox_raw, 
                            id_vars = ['ROUTE_NUMBER'],
                            value_vars = ['ON_FAREBOX_AM', 'ON_FAREBOX_Midday', 'ON_FAREBOX_PM Peak', 'ON_FAREBOX_PM Late'],
                            value_name = 'ON')
MTS_farebox_count['TIME_PERIOD'] = MTS_farebox_count['variable'].str.split('_').str.get(2)
MTS_farebox_count['TIME_PERIOD'] = MTS_farebox_count['TIME_PERIOD'].map(time_periods_remap)
MTS_farebox_count['DIRECTION'] = '00'  # add a generic 'DIRECTION' column to match APC targets
MTS_farebox_count

,ROUTE_NUMBER,variable,ON,TIME_PERIOD,DIRECTION
0,888,ON_FAREBOX_AM,0.000000,AM,00
1,891,ON_FAREBOX_AM,0.666667,AM,00
2,892,ON_FAREBOX_AM,1.500000,AM,00
3,906/907,ON_FAREBOX_AM,693.887097,AM,00
4,972,ON_FAREBOX_AM,5.370968,AM,00
5,973,ON_FAREBOX_AM,13.580645,AM,00
6,974,ON_FAREBOX_AM,16.258065,AM,00
7,978,ON_FAREBOX_AM,9.564516,AM,00
8,979,ON_FAREBOX_AM,12.887097,AM,00
9,888,ON_FAREBOX_Midday,1.680000,MIDDAY,00


In [23]:
MTS_farebox_count['ROUTE_NUMBER'].unique()

array(['888', '891', '892', '906/907', '972', '973', '974', '978', '979'],
      dtype=object)

In [24]:
NCTD_APC_data = pd.read_excel(os.path.join(data_folder, NCTD_APC_file), 
                              sheet_name = sheet_NCTD_APC,
                              usecols = NCTD_apc_keep_cols + ['stop_id'],
                              engine='openpyxl')
NCTD_APC_data['DIRECTION'] = NCTD_APC_data['ETC_STOP_ID'].str.split('_').str.get(3)

In [25]:
NCTD_APC_count = pd.melt(NCTD_APC_data, 
                         id_vars = ['route_short_name', 'DIRECTION', 'stop_id'],
                         value_vars = NCTD_apc_keep_cols[2:])
NCTD_APC_count['variable'] = NCTD_APC_count['variable'].str.replace('EXPAND_','')
NCTD_APC_count['variable'] = NCTD_APC_count['variable'].str.replace('NORMALIZED_','')
NCTD_APC_count[['variable', 'TIME_PERIOD']] = NCTD_APC_count['variable'].str.split('_', expand = True)
NCTD_APC_count['TIME_PERIOD'] = NCTD_APC_count['TIME_PERIOD'].map(time_periods_remap)
NCTD_APC_count = NCTD_APC_count.rename(columns = {
    'route_short_name': 'ROUTE_NUMBER',
    'direction_id': 'DIRECTION'})
NCTD_APC_count

,ROUTE_NUMBER,DIRECTION,stop_id,variable,value,TIME_PERIOD
0,398,00,28007,ON,85.475635,AM
1,398,00,28006,ON,70.391699,AM
2,398,00,28005,ON,2.513989,AM
3,398,00,28004,ON,12.569946,AM
4,398,00,28003,ON,7.541968,AM
...,...,...,...,...,...,...
20555,652,00,20514,OFF,0.000000,EVE
20556,652,00,20513,OFF,0.000000,EVE
20557,652,00,22353,OFF,0.000000,EVE
20558,652,00,20754,OFF,0.000000,EVE


In [26]:
NCTD_APC_count = pd.pivot_table(NCTD_APC_count, 
                                values="value", 
                                index=['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD', 'stop_id'],
                                columns = ['variable'], 
                                aggfunc=np.sum, 
                                fill_value=0)
NCTD_APC_count = NCTD_APC_count.reset_index()

In [29]:
APC_combined = pd.concat([MTS_APC_data, NCTD_APC_count])
APC_combined['ROUTE_NUMBER'] = APC_combined['ROUTE_NUMBER'].astype('str')

In [30]:
APC_combined['ROUTE_NUMBER'].unique()

array(['1', '10', '105', '11', '110', '115', '12', '120', '13', '14',
       '18', '2', '20', '201', '202', '204', '215', '225', '235', '237',
       '25', '27', '28', '280', '290', '3', '30', '31', '35', '4', '41',
       '43', '44', '5', '6', '60', '7', '701', '704', '705', '707', '709',
       '712', '8', '815', '816', '83', '832', '833', '834', '838', '84',
       '848', '851', '852', '854', '855', '856', '864', '872', '874',
       '875', '88', '894', '9', '901', '904', '905', '909', '916', '917',
       '921', '923', '928', '929', '932', '933', '934', '936', '944',
       '945', '950', '955', '961', '962', '963', '964', '965', '967',
       '968', '985', '992', 'Blue', 'Green', 'Orange', '101', '302',
       '303', '304', '305', '306', '308', '309', '311', '313', '315',
       '318', '323', '325', '332', '334', '347', '350', '351', '352',
       '353', '354', '355', '356', '357', '358', '359', '371', '388',
       '392', '395', '398', '399', '444', '445', '604', '608', '609',
   

In [68]:
MTS_farebox_count.to_csv(os.path.join(output_folder, 'checks', 'MTS_farebox_debug.csv'), index=False)

In [31]:
all_counts = pd.concat([APC_combined, MTS_farebox_count])
all_counts

,ROUTE_NUMBER,DIRECTION,TIME_PERIOD,stop_id,OFF,ON,variable
0,1,00,AM,10106.0,0.278508,2.106445,NaN
1,1,00,AM,10111.0,0.167105,1.330387,NaN
2,1,00,AM,10114.0,0.278508,0.443462,NaN
3,1,00,AM,10129.0,0.501315,2.051013,NaN
4,1,00,AM,10139.0,0.779823,7.594290,NaN
...,...,...,...,...,...,...,...
31,972,00,EVE,NaN,NaN,0.000000,ON_FAREBOX_PM Late
32,973,00,EVE,NaN,NaN,0.000000,ON_FAREBOX_PM Late
33,974,00,EVE,NaN,NaN,0.000000,ON_FAREBOX_PM Late
34,978,00,EVE,NaN,NaN,0.000000,ON_FAREBOX_PM Late


In [32]:
#counts_data = get_count()
#count == 2019 counts by default
#    (remove references to 2019 counts. If need to switch to 2020, the name space will be less confusing)

key1_groupby_fields = ['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD']
key2_groupby_fields = ['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD', 'stop_id']
key2_rts = ['398', '399', 'Blue', 'Green', 'Orange']

agg_scheme = {
    'ON': 'sum',
    'OFF': 'sum',
}

counts_by_key1 = APC_combined.groupby(key1_groupby_fields).agg({'ON': 'sum'})
counts_by_key1 = counts_by_key1.reset_index()

# append farebox data
counts_by_key1 = pd.concat([counts_by_key1, MTS_farebox_count[['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD', 'ON']]])

counts_by_key2 = APC_combined[APC_combined["ROUTE_NUMBER"].isin(key2_rts)]
counts_by_key2 = counts_by_key2.groupby(key2_groupby_fields).agg(agg_scheme)
counts_by_key2 = counts_by_key2.reset_index()

for targets, fields in zip((counts_by_key1, counts_by_key2), (key1_groupby_fields, key2_groupby_fields)):
    
    targets['key'] = targets[fields[0]].astype('str')
    for fld in fields[1:]:
        targets['key'] = targets['key'] + "_" + targets[fld].astype('str')
    targets = targets.rename(columns = rename_cols)

In [31]:
counts_by_key1.to_csv(os.path.join(output_folder, 'checks', 'counts_by_key1_debug.csv'), index=False)

In [33]:
rename_cols = {
    'ROUTE_NUMBER': 'Route',
    'DIRECTION': 'Dir',
    'ON': 'ONs',
    'OFF': 'OFFs'    
}

In [34]:
counts_by_key1 = counts_by_key1.rename(columns = rename_cols)
counts_by_key1

,Route,Dir,TIME_PERIOD,ONs,key
0,1,00,AM,229.657982,1_00_AM
1,1,00,EVE,137.895732,1_00_EVE
2,1,00,MIDDAY,553.548816,1_00_MIDDAY
3,1,00,PM,469.580074,1_00_PM
4,1,01,AM,288.971050,1_01_AM
...,...,...,...,...,...
31,972,00,EVE,0.000000,972_00_EVE
32,973,00,EVE,0.000000,973_00_EVE
33,974,00,EVE,0.000000,974_00_EVE
34,978,00,EVE,0.000000,978_00_EVE


In [35]:
counts_by_key2 = counts_by_key2.rename(columns = rename_cols)
counts_by_key2

,Route,Dir,TIME_PERIOD,stop_id,ONs,OFFs,key
0,398,00,AM,28000,0.000000,50.279785,398_00_AM_28000
1,398,00,AM,28001,5.027979,40.223828,398_00_AM_28001
2,398,00,AM,28002,2.513989,35.195850,398_00_AM_28002
3,398,00,AM,28003,7.541968,37.709839,398_00_AM_28003
4,398,00,AM,28004,12.569946,7.541968,398_00_AM_28004
...,...,...,...,...,...,...,...
839,Orange,01,PM,75090,56.878722,206.995348,Orange_01_PM_75090
840,Orange,01,PM,75092,88.318015,131.037055,Orange_01_PM_75092
841,Orange,01,PM,75102,271.073908,730.903574,Orange_01_PM_75102
842,Orange,01,PM,75109,0.000000,136.436934,Orange_01_PM_75109


### Combine Survey "Unique IDs" with Counts Targets 

In [36]:
#--- Prep Survey groupings targets
#--- assemble1 for line targets
#--- assemble2 & assemble3 for station ons and offs
#--------------------------------------------------
asmbl1_fields = ['ROUTE_NUMBER', 'DIRECTION', 'TIME_PERIOD', 'subkey1', 'subkey2', 'subkey2_Ons', 'subkey2_Offs']
assemble1 = survey_2023[cols + ['ROUTE_NUMBER']].groupby('UID')[asmbl1_fields].agg('first')
assemble1 = assemble1.reset_index()

assemble2 = assemble1.copy()
assemble3 = assemble1.copy()

#
#--- Survey groupings for line boardings targets
#-----------------------------------------------
assemble1 = pd.merge(left = assemble1, right = counts_by_key1[['ONs', 'key']],
                     how = 'left',
                     left_on = 'subkey1', right_on = 'key',
                    )
assemble1 = assemble1.rename(columns={'ONs': 'targets'})

#
#--- Survey groupings for station on targets
#-------------------------------------------
assemble2 = assemble2[assemble2['subkey2']!=""]
assemble2 = pd.merge(left = assemble2, right = counts_by_key2[['ONs', 'key']],
                     how = 'left',
                     left_on = 'subkey2_Ons', right_on = 'key',
                     #suffixes = ('', '2')
                    )
assemble2['key'] = assemble2['key'] + '_Ons'
assemble2 = assemble2.rename(columns={'ONs': 'targets'})

#
#--- Survey groupings for station off targets
#-------------------------------------------
assemble3 = assemble3[assemble3['subkey2']!=""]
assemble3 = pd.merge(left = assemble3, right = counts_by_key2[['OFFs', 'key']],
                     how = 'left',
                     left_on = 'subkey2_Offs', right_on = 'key',
                     #suffixes = ('', '2')
                    )
assemble3['key'] = assemble3['key'] + '_Offs'
assemble3 = assemble3.rename(columns={'OFFs': 'targets'})

#assemble_inputs['unique_weight_id'] = assemble_inputs['key1']
#assemble_inputs['identifier'] = 1
assemble1

,UID,ROUTE_NUMBER,DIRECTION,TIME_PERIOD,subkey1,subkey2,subkey2_Ons,subkey2_Offs,targets,key
0,101_00_AM,101,00,AM,101_00_AM,,None,None,209.630000,101_00_AM
1,101_00_EVE,101,00,EVE,101_00_EVE,,None,None,136.390000,101_00_EVE
2,101_00_MIDDAY,101,00,MIDDAY,101_00_MIDDAY,,None,None,483.630000,101_00_MIDDAY
3,101_00_PM,101,00,PM,101_00_PM,,None,None,244.690000,101_00_PM
4,101_01_AM,101,01,AM,101_01_AM,,None,None,177.850000,101_01_AM
...,...,...,...,...,...,...,...,...,...,...
5895,Orange_01_PM_75102_75109,Orange,01,PM,Orange_01_PM,Orange_01_PM_75102_75109,Orange_01_PM_75102,Orange_01_PM_75109,2452.984873,Orange_01_PM
5896,Orange_01_PM_99887_75039,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75039,Orange_01_PM_99887,Orange_01_PM_75039,2452.984873,Orange_01_PM
5897,Orange_01_PM_99887_75066,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75066,Orange_01_PM_99887,Orange_01_PM_75066,2452.984873,Orange_01_PM
5898,Orange_01_PM_99887_75072,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75072,Orange_01_PM_99887,Orange_01_PM_75072,2452.984873,Orange_01_PM


In [37]:
assemble1.to_csv(os.path.join(output_folder, 'checks', 'assembel1_debug.csv'), index=False)

In [38]:
assemble2 = assemble2[assemble2['subkey2']!=""]#_inputs[assemble_inputs['ONs2']>0].sort_values('ONs2')
assemble2

,UID,ROUTE_NUMBER,DIRECTION,TIME_PERIOD,subkey1,subkey2,subkey2_Ons,subkey2_Offs,targets,key
0,398_00_AM_28006_28000,398,00,AM,398_00_AM,398_00_AM_28006_28000,398_00_AM_28006,398_00_AM_28000,70.391699,398_00_AM_28006_Ons
1,398_00_AM_28006_28001,398,00,AM,398_00_AM,398_00_AM_28006_28001,398_00_AM_28006,398_00_AM_28001,70.391699,398_00_AM_28006_Ons
2,398_00_AM_28006_28002,398,00,AM,398_00_AM,398_00_AM_28006_28002,398_00_AM_28006,398_00_AM_28002,70.391699,398_00_AM_28006_Ons
3,398_00_AM_28006_28003,398,00,AM,398_00_AM,398_00_AM_28006_28003,398_00_AM_28006,398_00_AM_28003,70.391699,398_00_AM_28006_Ons
4,398_00_AM_28006_28004,398,00,AM,398_00_AM,398_00_AM_28006_28004,398_00_AM_28006,398_00_AM_28004,70.391699,398_00_AM_28006_Ons
...,...,...,...,...,...,...,...,...,...,...
5047,Orange_01_PM_75102_75109,Orange,01,PM,Orange_01_PM,Orange_01_PM_75102_75109,Orange_01_PM_75102,Orange_01_PM_75109,271.073908,Orange_01_PM_75102_Ons
5048,Orange_01_PM_99887_75039,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75039,Orange_01_PM_99887,Orange_01_PM_75039,0.359992,Orange_01_PM_99887_Ons
5049,Orange_01_PM_99887_75066,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75066,Orange_01_PM_99887,Orange_01_PM_75066,0.359992,Orange_01_PM_99887_Ons
5050,Orange_01_PM_99887_75072,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75072,Orange_01_PM_99887,Orange_01_PM_75072,0.359992,Orange_01_PM_99887_Ons


In [39]:
assemble3 = assemble3[assemble2['subkey2']!=""]#_inputs[assemble_inputs['ONs2']>0].sort_values('ONs2')
assemble3

,UID,ROUTE_NUMBER,DIRECTION,TIME_PERIOD,subkey1,subkey2,subkey2_Ons,subkey2_Offs,targets,key
0,398_00_AM_28006_28000,398,00,AM,398_00_AM,398_00_AM_28006_28000,398_00_AM_28006,398_00_AM_28000,50.279785,398_00_AM_28000_Offs
1,398_00_AM_28006_28001,398,00,AM,398_00_AM,398_00_AM_28006_28001,398_00_AM_28006,398_00_AM_28001,40.223828,398_00_AM_28001_Offs
2,398_00_AM_28006_28002,398,00,AM,398_00_AM,398_00_AM_28006_28002,398_00_AM_28006,398_00_AM_28002,35.195850,398_00_AM_28002_Offs
3,398_00_AM_28006_28003,398,00,AM,398_00_AM,398_00_AM_28006_28003,398_00_AM_28006,398_00_AM_28003,37.709839,398_00_AM_28003_Offs
4,398_00_AM_28006_28004,398,00,AM,398_00_AM,398_00_AM_28006_28004,398_00_AM_28006,398_00_AM_28004,7.541968,398_00_AM_28004_Offs
...,...,...,...,...,...,...,...,...,...,...
5047,Orange_01_PM_75102_75109,Orange,01,PM,Orange_01_PM,Orange_01_PM_75102_75109,Orange_01_PM_75102,Orange_01_PM_75109,136.436934,Orange_01_PM_75109_Offs
5048,Orange_01_PM_99887_75039,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75039,Orange_01_PM_99887,Orange_01_PM_75039,160.196400,Orange_01_PM_75039_Offs
5049,Orange_01_PM_99887_75066,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75066,Orange_01_PM_99887,Orange_01_PM_75066,92.517921,Orange_01_PM_75066_Offs
5050,Orange_01_PM_99887_75072,Orange,01,PM,Orange_01_PM,Orange_01_PM_99887_75072,Orange_01_PM_99887,Orange_01_PM_75072,72.478371,Orange_01_PM_75072_Offs


In [40]:
assemble = pd.concat([assemble1, assemble2, assemble3])#_inputs.sort_values('subkey2')
assemble['identifier'] = 1
incidence_mat = assemble.pivot_table(values='identifier', index='UID', columns='key').fillna(0).astype(int)
incidence_mat

key,101_00_AM,101_00_EVE,101_00_MIDDAY,101_00_PM,101_01_AM,101_01_EVE,101_01_MIDDAY,101_01_PM,105_00_AM,105_00_EVE,...,Orange_01_PM_75088_Ons,Orange_01_PM_75090_Offs,Orange_01_PM_75090_Ons,Orange_01_PM_75092_Offs,Orange_01_PM_75092_Ons,Orange_01_PM_75102_Offs,Orange_01_PM_75102_Ons,Orange_01_PM_75109_Offs,Orange_01_PM_99887_Offs,Orange_01_PM_99887_Ons
UID,,,,,,,,,,,,,,,,,,,,,
101_00_AM,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
101_00_EVE,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
101_00_MIDDAY,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
101_00_PM,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
101_01_AM,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Orange_01_PM_75102_75109,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0
Orange_01_PM_99887_75039,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
Orange_01_PM_99887_75066,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [41]:
#--- Write unique_wts, incidence_matrix and obs_target csv files

#1. unique_wts_id
pd.Series(incidence_mat.index.rename("unique_weight_id")).to_csv(os.path.join(output_folder, unique_wts_filename), index=False)

#2. Incidence matrix
incidence_mat.to_csv(os.path.join(output_folder, incidence_matrix_filename), index=False)

#3. Obs targets file
targets = assemble.groupby('key').agg({'targets': min})#loc[assemble['key']==(list(incidence_mat.columns)), ['key', 'ONs']]
targets = targets.reset_index()
targets.to_csv(os.path.join(output_folder, targets_filesname), index=False)

In [83]:
incidence_mat.index.rename("unique_weight_id")

Index(['101_00_AM', '101_00_EVE', '101_00_MIDDAY', '101_00_PM', '101_01_AM',
       '101_01_EVE', '101_01_MIDDAY', '101_01_PM', '105_00_AM', '105_00_EVE',
       ...
       'Orange_01_PM_75092_75109', 'Orange_01_PM_75102_75087',
       'Orange_01_PM_75102_75088', 'Orange_01_PM_75102_75090',
       'Orange_01_PM_75102_75092', 'Orange_01_PM_75102_75109',
       'Orange_01_PM_99887_75039', 'Orange_01_PM_99887_75066',
       'Orange_01_PM_99887_75072', 'Orange_01_PM_99887_75075'],
      dtype='object', name='unique_weight_id', length=5900)

In [42]:
#5. Survey Records
#survey_2023['ROUTE_NUMBER'] = survey_2023['ROUTE_SURVEYED_CODE'].str.slice(start=1, stop=4).astype('int')
survey_2023['unique_weight_id'] = survey_2023['UID']
cols = (
    srvy_use_cols
    + ['BOARDING_ID', 'ALIGHTING_ID', 'UID', 'subkey1', 'subkey2', 'subkey2_Ons', 'subkey2_Offs', 'unique_weight_id']
)

# To avoid keeping some deleted routes survey records 
survey_2023_for_analysis = survey_2023[cols].copy()
survey_2023_for_analysis = survey_2023_for_analysis[survey_2023_for_analysis['UID'].isin(list(incidence_mat.index))]
survey_2023_for_analysis.to_csv(os.path.join(output_folder, srvy_filename))